# Data cleaning & missing-value handling — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def rate(x): return float(np.mean(np.asarray(x)))
def nearest_predict(train_x, train_y, test_x):
    train_x=np.asarray(train_x); train_y=np.asarray(train_y); preds=[]
    for t in np.asarray(test_x):
        preds.append(train_y[int(np.argmin(np.abs(train_x-t)))])
    return np.asarray(preds)
print('setup ok')

## Handling missingness without lying to the model
Missing-value handling is a modeling decision about what information remains after cleaning. These examples compare mean imputation, segment-aware imputation, indicators, dropping rows, and a sensitivity band.

In [ ]:
# 1) Mean imputation preserves the observed mean on MCAR data
x=np.array([10.,12.,np.nan,18.,20.]); mu=np.nanmean(x); filled=np.where(np.isnan(x),mu,x)
plt.figure(figsize=(6,3)); plt.bar(range(len(filled)),filled,color=['tab:red' if np.isnan(v) else 'tab:blue' for v in x]); plt.axhline(mu,c='k',ls='--'); plt.title(f'observed mean = imputed value = {mu:.1f}')
assert abs(mu-15.0)<1e-12 and abs(filled.mean()-15.0)<1e-12

In [ ]:
# 2) Group means avoid washing out segment structure
g=np.array(['A','A','A','B','B','B']); x=np.array([10.,12.,np.nan,30.,32.,np.nan]); filled=x.copy()
for grp in ['A','B']:
    filled[(g==grp)&np.isnan(x)]=np.nanmean(x[g==grp])
plt.figure(figsize=(6,3)); plt.bar(range(6),filled); plt.title('missing values filled within each group')
assert np.allclose(filled,[10,12,11,30,32,31])

In [ ]:
# 3) A missingness indicator keeps the fact that a value was absent
x=np.array([10.,12.,np.nan,18.,20.]); ind=np.isnan(x).astype(int)
plt.figure(figsize=(6,3)); plt.step(range(len(ind)),ind,where='mid'); plt.ylim(-.1,1.1); plt.title('indicator separates value from missingness')
assert ind.sum()==1 and ind[2]==1

In [ ]:
# 4) Dropping rows can change the target rate when missingness is informative
y=np.array([0,0,1,1,1,1]); observed=np.array([1,1,1,1,0,0]).astype(bool)
plt.figure(figsize=(6,3)); plt.bar(['full','after drop'],[y.mean(),y[observed].mean()]); plt.ylim(0,1); plt.title('complete-case analysis changes the label mix')
assert abs(y.mean()-2/3)<1e-12 and y[observed].mean()==0.5

In [ ]:
# 5) A sensitivity band asks how much conclusions depend on the fill value
x=np.array([10.,12.,np.nan,18.,20.]); fills=np.linspace(8,22,50); means=[np.where(np.isnan(x),f,x).mean() for f in fills]
plt.figure(figsize=(6,3)); plt.plot(fills,means); plt.xlabel('assumed missing value'); plt.ylabel('completed mean'); plt.title('imputation is an assumption you can sweep')
assert abs(means[25]-15.0285714286)<1e-9